In [12]:
import numpy as np
import os

def generate_lid_driven_cavity(nx, ny, Lx=1.0, Ly=1.0,
                                lid_u=1.0, lid_v=0.0,
                                output_dir="cavity_mesh"):
    os.makedirs(output_dir, exist_ok=True)

    # ── 1. params.txt ────────────────────────────────────────────────
    with open(os.path.join(output_dir, "params.txt"), "w") as f:
        f.write(f"{nx} {ny}\n")

    # ── 2. 节点坐标  (ny+1)×(nx+1) ──────────────────────────────────
    xs = np.linspace(0.0, Lx, nx + 1)
    ys = np.linspace(0.0, Ly, ny + 1)
    X, Y = np.meshgrid(xs, ys)
    np.savetxt(os.path.join(output_dir, "x.dat"), X, fmt="%.10f")
    np.savetxt(os.path.join(output_dir, "y.dat"), Y, fmt="%.10f")

    # ── 3. bctype  ny×nx ─────────────────────────────────────────────
    #   i=0    → 底壁（无滑移）
    #   i=ny-1 → 顶盖（固定速度 -2）
    #   j=0    → 左壁（无滑移）
    #   j=nx-1 → 右壁（无滑移）
    bctype = np.zeros((ny, nx), dtype=int)
    bctype[0,  :]  =  1    # 顶壁：无滑移
    bctype[:,  0]  =  1    # 左壁：无滑移
    bctype[:, -1]  =  1    # 右壁：无滑移
    bctype[-1, :]  =  1    # 底盖：固定速度（覆盖角点）
    np.savetxt(os.path.join(output_dir, "bctype.dat"), bctype, fmt="%d")

    # ── 4. zoneid  ny×nx ─────────────────────────────────────────────
    #   顶盖单元 → zone 1
    #   其余     → zone 0
    zoneid = np.zeros((ny, nx), dtype=int)
    zoneid[0,  :] = 1      # 顶盖行：zone 1
    np.savetxt(os.path.join(output_dir, "zoneid.dat"), zoneid, fmt="%d")

    # ── 5. zoneuv.txt ─────────────────────────────────────────────────
    #   行号 = zone id
    #   zone 0：内部/壁面，速度为 0
    #   zone 1：顶盖，速度为 lid_u / lid_v
    with open(os.path.join(output_dir, "zoneuv.txt"), "w") as f:
        f.write(f"0.0000000000  0.0000000000\n")          # zone 0
        f.write(f"{lid_u:.10f}  {lid_v:.10f}\n")          # zone 1

    # ── 6. 打印摘要 ───────────────────────────────────────────────────
    print(f"网格已生成：{output_dir}/")
    print(f"  节点矩阵  : ({ny+1}) × ({nx+1})")
    print(f"  单元矩阵  : ({ny})   × ({nx})")
    print(f"  顶盖速度  : u={lid_u}, v={lid_v}")
    print(f"  内部点数  : {int((bctype ==  0).sum())}")
    print(f"  顶盖单元数: {int((bctype == -2).sum())}  (zone 1)")
    print(f"  壁面单元数: {int((bctype ==  1).sum())}  (zone 0)")


if __name__ == "__main__":
    generate_lid_driven_cavity(
        nx=64, ny=64,
        Lx=1.0, Ly=1.0,
        lid_u=1.0, lid_v=0.0,
        output_dir="ldc_uni"
    )

网格已生成：ldc_uni/
  节点矩阵  : (65) × (65)
  单元矩阵  : (64)   × (64)
  顶盖速度  : u=1.0, v=0.0
  内部点数  : 3844
  顶盖单元数: 0  (zone 1)
  壁面单元数: 252  (zone 0)


In [13]:
import numpy as np
import os

def exp_stretch(n, L, alpha=5.0):
    """
    在两端（0 和 L）附近指数加密，中间稀疏。
    alpha 越大，壁面处越密集。
    原理：将均匀参数 t∈[0,1] 分两段做 e^x 映射，
         左半段 [0,0.5] 从 0 映射到 L/2，右半段对称。
    """
    t = np.linspace(0.0, 1.0, n + 1)
    x = np.empty(n + 1)
    half = (np.exp(alpha * 0.5) - 1.0)          # 归一化分母
    for i, ti in enumerate(t):
        if ti <= 0.5:
            x[i] = L / 2.0 * (np.exp(alpha * ti) - 1.0) / half
        else:
            x[i] = L - L / 2.0 * (np.exp(alpha * (1.0 - ti)) - 1.0) / half
    return x


def generate_lid_driven_cavity(nx, ny, Lx=1.0, Ly=1.0,
                                lid_u=1.0, lid_v=0.0,
                                output_dir="cavity_mesh",
                                alpha_x=5.0, alpha_y=5.0):
    os.makedirs(output_dir, exist_ok=True)

    # ── 1. params.txt ────────────────────────────────────────────────
    with open(os.path.join(output_dir, "params.txt"), "w") as f:
        f.write(f"{nx} {ny}\n")

    # ── 2. 节点坐标  (ny+1)×(nx+1) ──────────────────────────────────
    # 【仅此处改动】用指数拉伸替换均匀 linspace
    xs = exp_stretch(nx, Lx, alpha=alpha_x)   # x 方向：左右壁面加密
    ys = exp_stretch(ny, Ly, alpha=alpha_y)   # y 方向：上下壁面加密
    X, Y = np.meshgrid(xs, ys)
    np.savetxt(os.path.join(output_dir, "x.dat"), X, fmt="%.10f")
    np.savetxt(os.path.join(output_dir, "y.dat"), Y, fmt="%.10f")

    # ── 3. bctype  ny×nx ─────────────────────────────────────────────
    bctype = np.zeros((ny, nx), dtype=int)
    bctype[0,  :]  =  1
    bctype[:,  0]  =  1
    bctype[:, -1]  =  1
    bctype[-1, :]  =  1
    np.savetxt(os.path.join(output_dir, "bctype.dat"), bctype, fmt="%d")

    # ── 4. zoneid  ny×nx ─────────────────────────────────────────────
    zoneid = np.zeros((ny, nx), dtype=int)
    zoneid[0,  :] = 1
    np.savetxt(os.path.join(output_dir, "zoneid.dat"), zoneid, fmt="%d")

    # ── 5. zoneuv.txt ─────────────────────────────────────────────────
    with open(os.path.join(output_dir, "zoneuv.txt"), "w") as f:
        f.write(f"0.0000000000  0.0000000000\n")
        f.write(f"{lid_u:.10f}  {lid_v:.10f}\n")

    # ── 6. 打印摘要 ───────────────────────────────────────────────────
    print(f"网格已生成：{output_dir}/")
    print(f"  节点矩阵  : ({ny+1}) × ({nx+1})")
    print(f"  单元矩阵  : ({ny})   × ({nx})")
    print(f"  顶盖速度  : u={lid_u}, v={lid_v}")
    print(f"  拉伸系数  : alpha_x={alpha_x}, alpha_y={alpha_y}")
    print(f"  x 首间距  : {xs[1]-xs[0]:.6f}  (均匀时 {Lx/nx:.6f})")
    print(f"  x 中间距  : {xs[nx//2+1]-xs[nx//2]:.6f}")
    print(f"  内部点数  : {int((bctype ==  0).sum())}")
    print(f"  壁面单元数: {int((bctype ==  1).sum())}")


if __name__ == "__main__":
    generate_lid_driven_cavity(
        nx=100, ny=100,
        Lx=1.0, Ly=1.0,
        lid_u=1.0, lid_v=0.0,
        output_dir="ldc_exp",
        alpha_x=5.0,   # 调大 → 壁面处更密，中间更稀
        alpha_y=5.0,
    )

网格已生成：ldc_exp/
  节点矩阵  : (101) × (101)
  单元矩阵  : (100)   × (100)
  顶盖速度  : u=1.0, v=0.0
  拉伸系数  : alpha_x=5.0, alpha_y=5.0
  x 首间距  : 0.002292  (均匀时 0.010000)
  x 中间距  : 0.026566
  内部点数  : 9604
  壁面单元数: 396
